# 06 · Hypothesis verdicts and export

Applies the pass/fail rules fixed in Section 3 of the design document mechanically to the saved tables, then zips
all tables (CSV + LaTeX), figures (PDF + PNG), logs and run manifests into `results/calvol_results_<mode>.zip`.

Development-period verdicts are for debugging only. Every pre-registered hypothesis is reported from the holdout run,
whether it passes or fails (Section 9).

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

In [ ]:
V = pp.stage_summary(cfg)
with pd.option_context("display.max_colwidth", 200):
    display(V)

## Files produced

In [ ]:
for sub in ["tables", "figures"]:
    fs = sorted((P.results / sub).glob("*"))
    print(f"{sub}: {len(fs)} files")
    print("  " + "\n  ".join(f.name for f in fs if f.suffix in (".csv", ".pdf")))

In [ ]:
z = pp.export_results(cfg)
print("Archive:", z, f"({z.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(z))